# **EDA: El impacto de la IA**
## Análisis de las hipótesis

En este notebook se comprueba cada hipótesis con los datos. Las hipótesis se plantearon antes de analizar y se verifica si se cumplen o si ocurre lo contrario.

#### **Hipótesis**:

1. Quien usa más IA, es más productivo.
2. El uso excesivo de la IA tiene efectos negativos en el bienestar.
3. Los estudiantes que usan más IA sacan mejores notas, pero retienen menos lo aprendido.

Las hipótesis 1 y 2 se comprueban con el dataset de empleados (la 2 también con el de estudiantes) y la 3 con el de estudiantes. Se parte de los ficheros originales y se repite la unión de los dos ficheros de empleados, que ya se documentó en el notebook de limpieza.

In [1]:
import pandas as pd

# Empleados: se unen los dos ficheros y se reordenan las columnas (igual que en el notebook de limpieza)
df_features = pd.read_csv("../data/ai_productivity_features.csv")
df_targets = pd.read_csv("../data/ai_productivity_targets.csv")
df_empleados = pd.merge(df_features, df_targets, on="Employee_ID")

columns_orden = ["productivity_score", "burnout_risk_score", "burnout_risk_level"]
columns_resto = [c for c in df_empleados.columns if c not in columns_orden]
df_empleados_2 = df_empleados[columns_resto + columns_orden]

# Estudiantes
df_estudiantes = pd.read_csv("../data/ai_student_impact_dataset.csv")

print(df_empleados_2.shape, df_estudiantes.shape)

(4500, 17) (50000, 16)


# Hipótesis 1: Quien usa más IA, es más productivo

##### Paso 1: Conocer las variables

Antes de buscar relaciones, se miran por separado las variables `ai_tool_usage_hours_per_week` y `productivity_score` con `describe()`.

In [2]:
df_empleados_2["ai_tool_usage_hours_per_week"].describe()

count    4500.000000
mean       10.349667
std         5.808240
min         0.000000
25%         6.000000
50%         9.700000
75%        14.300000
max        30.400000
Name: ai_tool_usage_hours_per_week, dtype: float64

**Horas de IA a la semana:**

La media es 10,3 h y la mediana 9,7 h. La mitad de las personas usa entre 6 y 14,3 horas. El mínimo es 0 y el máximo 30,4, así que hay quien no la usa y unos pocos que la usan mucho.

In [3]:
df_empleados_2["productivity_score"].describe()

count    4500.000000
mean       64.946600
std        14.297129
min        20.500000
25%        54.975000
50%        64.800000
75%        74.900000
max       100.000000
Name: productivity_score, dtype: float64

**Productividad:**

Va de 20,5 a 100, con media (64,9) y mediana (64,8) casi iguales. Los valores están bien repartidos y casi nadie está en el máximo.

**Conclusión:** Hay variación suficiente en las dos variables para comprobar si se relacionan.

##### Paso 2: Correlación

La correlación mide si dos variables suben y bajan a la vez. Va de -1 a 1. Cerca de 0 no hay relación y cuanto más se acerca a 1 o a -1, más fuerte es.

Se calcula primero la correlación de todas las variables numéricas con la productividad.

In [4]:
df_empleados_2.corr(numeric_only=True)["productivity_score"].sort_values()

meeting_hours_per_week         -0.445299
manual_work_hours_per_week     -0.439878
burnout_risk_score             -0.392000
collaboration_hours_per_week   -0.364280
error_rate_percent             -0.295577
task_complexity_score           0.005009
learning_time_hours_per_week    0.112890
work_life_balance_score         0.403521
experience_years                0.463643
ai_tool_usage_hours_per_week    0.498452
focus_hours_per_day             0.514713
tasks_automated_percent         0.533995
productivity_score              1.000000
Name: productivity_score, dtype: float64

Las tareas automatizadas tienen una correlación parecida a las horas de IA. Se comprueba si se parecen entre sí.

In [5]:
df_empleados_2[["ai_tool_usage_hours_per_week", "tasks_automated_percent"]].corr()

,ai_tool_usage_hours_per_week,tasks_automated_percent
ai_tool_usage_hours_per_week,1.000000,0.947383
tasks_automated_percent,0.947383,1.000000


Se mira también con qué otras variables se mueven las horas de IA: el trabajo manual y las reuniones.

In [6]:
df_empleados_2[["ai_tool_usage_hours_per_week", "manual_work_hours_per_week", "meeting_hours_per_week"]].corr()

,ai_tool_usage_hours_per_week,manual_work_hours_per_week,meeting_hours_per_week
ai_tool_usage_hours_per_week,1.000000,-0.634209,-0.271238
manual_work_hours_per_week,-0.634209,1.000000,0.173701
meeting_hours_per_week,-0.271238,0.173701,1.000000


##### Conclusiones del paso 2

- La relación entre horas de IA (`ai_tool_usage_hours_per_week`) y productividad (`productivity_score`) es positiva y moderada (0,50). Quien usa más IA tiende a ser más productivo, pero no es una relación exacta.
- Las horas de IA no son lo que más se relaciona con la productividad. Otras variables tienen valores parecidos o mayores:
    - Las tareas automatizadas (0,53) y las horas de foco (0,51).
    - Las tareas automatizadas se mueven casi a la vez que las horas de IA (0,95), por lo que miden prácticamente lo mismo.
    - Las reuniones (-0,45) y el trabajo manual (-0,44) van en sentido contrario.
    - La complejidad de las tareas no tiene relación (0,005).
- Quienes usan más IA dedican menos horas al trabajo manual (-0,63) y a reuniones (-0,27), lo que también puede influir en su productividad.
- La correlación solo dice que hay relación, no que la IA la cause. Todavía no se puede afirmar que la IA sea la responsable de la mayor productividad.
- Para ver cómo cambia la productividad al usar más IA, se comparan grupos en los pasos siguientes.

##### Paso 3: Crear los tramos de uso de IA

Los grupos muestran cómo cambia la productividad: si sube siempre o se frena.

Con `qcut()` se dividen las 4.500 personas en 4 grupos según sus horas de IA, para que los grupos tengan un tamaño parecido. Se añade una columna `tramo_ia` al dataset.

In [7]:
df_empleados_2["tramo_ia"] = pd.qcut(df_empleados_2["ai_tool_usage_hours_per_week"], q=4, labels=["Bajo", "Medio-bajo", "Medio-alto", "Alto"])

df_empleados_2[["ai_tool_usage_hours_per_week", "tramo_ia"]].head()

,ai_tool_usage_hours_per_week,tramo_ia
0,11.8,Medio-alto
1,10.8,Medio-alto
2,25.9,Alto
3,7.9,Medio-bajo
4,8.6,Medio-bajo


Los tramos son:

- **Bajo:** hasta 6 h a la semana
- **Medio-bajo:** más de 6 y hasta 9,7 h
- **Medio-alto:** más de 9,7 y hasta 14,3 h
- **Alto:** más de 14,3 h

Cada tramo incluye su límite superior: quien tiene exactamente 6,0 h está en Bajo.

##### Cuántas personas hay por tramo

In [8]:
horas = df_empleados_2["ai_tool_usage_hours_per_week"]

print("Exactamente 6,0 h:", (horas == 6.0).sum())
print("Exactamente 9,7 h:", (horas == 9.7).sum())
print("Exactamente 14,3 h:", (horas == 14.3).sum())
print("Menos de 6,0 h:", (horas < 6.0).sum())
print("Más de 14,3 h:", (horas > 14.3).sum())

Exactamente 6,0 h: 24
Exactamente 9,7 h: 26
Exactamente 14,3 h: 26
Menos de 6,0 h: 1119
Más de 14,3 h: 1103


In [9]:
df_empleados_2["tramo_ia"].value_counts().sort_index()

tramo_ia
Bajo          1143
Medio-bajo    1110
Medio-alto    1144
Alto          1103
Name: count, dtype: int64

Si los cuatro tramos fueran exactos, cada uno tendría 1.125 personas (4.500 ÷ 4). Tienen entre 1.103 y 1.144 porque hay personas con justo las horas de los cortes (24 con 6,0 h, 26 con 9,7 h y 26 con 14,3 h), que quedan en el tramo inferior. Por ejemplo, Bajo suma 1.119 personas con menos de 6 h y 24 con exactamente 6 h, es decir, 1.143.

Los tamaños son parecidos, así que se pueden comparar.

##### Paso 4: Productividad media por tramo

In [10]:
# media de productividad y número de personas de cada tramo
resumen_prod = df_empleados_2.groupby("tramo_ia")["productivity_score"].agg(["mean", "count"])
resumen_prod

,mean,count
tramo_ia,,
Bajo,55.865879,1143
Medio-bajo,62.080901,1110
Medio-alto,67.492483,1144
Alto,74.600000,1103


In [11]:
# cuánto sube la media al pasar de un tramo al siguiente
medias_prod = resumen_prod["mean"]
medias_prod.diff().round(1)

tramo_ia
Bajo          NaN
Medio-bajo    6.2
Medio-alto    5.4
Alto          7.1
Name: mean, dtype: float64

In [12]:
# diferencia de productividad entre el tramo Alto y el Bajo
dif_prod = medias_prod.loc["Alto"] - medias_prod.loc["Bajo"]

# diferencia en porcentaje respecto al punto de partida (Bajo)
pct_prod = dif_prod / medias_prod.loc["Bajo"] * 100

print("Diferencia:", round(dif_prod, 1))
print("Porcentaje:", round(pct_prod, 1))

Diferencia: 18.7
Porcentaje: 33.5


- La productividad sube en cada tramo: 55,9, 62,1, 67,5 y 74,6.
- Entre el grupo que menos usa IA y el que más hay **18,7 puntos de diferencia** (un 33,5 % más).
- De un tramo al siguiente sube 6,2, 5,4 y 7,1 puntos, y sigue aumentando al llegar al grupo Alto.
- Por tanto, más horas de IA se asocian con más productividad en todo el rango.

##### Paso 5: ¿Se mantiene la relación dentro de cada puesto?

Los puestos podrían usar la IA de forma distinta. Si fuera así, la relación anterior podría deberse al puesto y no a la IA. Se mira el uso medio de IA y la productividad media de cada puesto.

In [13]:
df_empleados_2.groupby("job_role")[["ai_tool_usage_hours_per_week", "productivity_score"]].mean()

,ai_tool_usage_hours_per_week,productivity_score
job_role,,
Analyst,9.866480,65.521188
Designer,5.194875,61.501524
Developer,15.397399,71.373812
Manager,5.053221,51.306135
Marketer,10.064590,66.144833
Writer,15.046855,71.266811


- Los puestos no usan la IA por igual: Developer (15,4 h) y Writer (15,0 h) la usan el triple que Manager (5,1 h) y Designer (5,2 h).
- Además, la productividad media del Manager es la más baja (51,3).

Por eso la relación anterior podría deberse al puesto y no a la IA. Para comprobarlo, se comparan los tramos dentro de cada puesto.

Con `pivot_table` se ponen los puestos en las filas, los tramos en las columnas y la productividad media en cada celda. Se lee fila a fila: si la productividad sube de izquierda a derecha, la relación se mantiene en ese puesto.

In [14]:
df_empleados_2.pivot_table(index="job_role", columns="tramo_ia", values="productivity_score", aggfunc="mean").round(1)

tramo_ia,Bajo,Medio-bajo,Medio-alto,Alto
job_role,,,,
Analyst,59.2,63.3,67.4,74.4
Designer,59.6,64.2,64.5,81.5
Developer,62.2,61.9,68.1,75.2
Manager,49.1,55.1,56.0,NaN
Marketer,59.8,64.1,68.2,71.8
Writer,65.0,64.0,69.0,74.1


Una media de pocas personas no es fiable, así que se cuenta cuántas hay en cada celda con `crosstab`.

In [15]:
pd.crosstab(index=df_empleados_2["job_role"], columns=df_empleados_2["tramo_ia"])

tramo_ia,Bajo,Medio-bajo,Medio-alto,Alto
job_role,,,,
Analyst,153,288,341,110
Designer,435,246,40,1
Developer,31,119,323,642
Manager,418,195,39,0
Marketer,94,211,265,88
Writer,12,51,136,262


En el puesto Manager no aparece nadie en el tramo Alto. Para entender por qué, se mira el máximo de horas de IA de cada puesto.

In [16]:
df_empleados_2.groupby("job_role")["ai_tool_usage_hours_per_week"].max()

job_role
Analyst      21.5
Designer     15.3
Developer    30.4
Manager      13.5
Marketer     22.5
Writer       27.1
Name: ai_tool_usage_hours_per_week, dtype: float64

##### Conclusiones del paso 5

- En Analyst, Developer, Marketer y Writer, el grupo Alto es más productivo que el Bajo (por ejemplo, Analyst pasa de 59,2 a 74,4 y Developer de 62,2 a 75,2). La relación se mantiene dentro del puesto y no se explica solo por él.
- En Manager no hay nadie en el tramo Alto. El NaN no es un dato perdido: el máximo de un manager es 13,5 h y el corte está en 14,3. Se compara hasta Medio-alto (de 49,1 a 56,0).
- En Designer solo hay 1 persona en Alto, así que ese dato no se interpreta.
- En Developer (31 personas en Bajo) y Writer (12) el primer escalón no sube, pero son grupos muy pequeños.

##### Paso 6: Más productivos gracias a la IA, pero ¿a costa de errores?

Se calcula el porcentaje medio de errores por tramo. La ficha de Kaggle no define esta variable; se interpreta como el porcentaje de errores en el trabajo.

In [17]:
medias_err = df_empleados_2.groupby("tramo_ia")["error_rate_percent"].mean()
medias_err

tramo_ia
Bajo          1.801085
Medio-bajo    1.989892
Medio-alto    2.140105
Alto          2.466908
Name: error_rate_percent, dtype: float64

In [18]:
dif_err = medias_err["Alto"] - medias_err["Bajo"]
pct_err = dif_err / medias_err["Bajo"] * 100

print("Errores en Bajo:", round(medias_err["Bajo"], 2))
print("Errores en Alto:", round(medias_err["Alto"], 2))
print("Diferencia:", round(dif_err, 2))
print("Porcentaje:", round(pct_err, 1))

Errores en Bajo: 1.8
Errores en Alto: 2.47
Diferencia: 0.67
Porcentaje: 37.0


- Los errores pasan del **1,80 % al 2,47 %**: 0,67 puntos más. El 37 % sale de 0,67 ÷ 1,80 × 100: un 37 % más.
- Los errores aumentan en cada tramo y más en el último (+0,19, +0,15 y +0,33).
- En proporción, los errores crecen algo más que la productividad (un 33,5 %).
- Los errores se relacionan con menos productividad (-0,30, en el paso 2), pero aun así el grupo con más errores es el más productivo.

##### Paso 7: Conclusión de la hipótesis 1

**La hipótesis se cumple** porque la productividad media sube en cada tramo de uso de IA (de 55,9 en Bajo a 74,6 en Alto) y la subida se mantiene dentro de los puestos con datos suficientes.

- Quienes usan IA más de 14,3 horas a la semana puntúan de media 74,6 sobre 100 en productividad. Quienes la usan 6 horas o menos, 55,9. La diferencia es de 18,7 puntos (un 33,5 % más).
- La subida es constante y se mantiene dentro de los puestos con datos suficientes, así que no se explica solo por el tipo de trabajo.
- Tiene un coste: el grupo que más IA usa también tiene más errores (2,47 % frente a 1,80 %, un 37 % más), aunque sigue siendo el más productivo.

**Limitaciones:**

- Que dos cosas vayan juntas no prueba que una cause la otra. Por ejemplo, quienes usan más IA dedican menos horas al trabajo manual (correlación de -0,63) y a reuniones (-0,27), y eso también puede influir en su productividad.
- Los datos son sintéticos y, según la descripción de Kaggle, simulan una productividad influida por el uso de IA. Por eso, el resultado es una reflexión sobre un escenario simulado, no evidencia sobre personas reales.

# Hipótesis 2: El uso excesivo de la IA tiene efectos negativos en el bienestar

- El bienestar se mide en los dos datasets: el de empleados y el de estudiantes.
- El uso excesivo es el tramo Alto.
- En empleados se mira `burnout_risk_score` y `work_life_balance_score`. El burnout está cortado en 10 (37 %), así que hay poca variación.
- En estudiantes se mira el burnout y la ansiedad.

## Dataset Empleados

##### Paso 1: Correlación

Sirve para ver si las horas de IA se mueven junto con el bienestar.

In [19]:
df_empleados_2[["ai_tool_usage_hours_per_week", "burnout_risk_score", "work_life_balance_score"]].corr()

,ai_tool_usage_hours_per_week,burnout_risk_score,work_life_balance_score
ai_tool_usage_hours_per_week,1.000000,0.037898,-0.063139
burnout_risk_score,0.037898,1.000000,-0.846410
work_life_balance_score,-0.063139,-0.846410,1.000000


##### Paso 2: Por tramo

Se compara el bienestar de los cuatro grupos que se crearon para la hipótesis 1.

In [20]:
df_empleados_2.groupby("tramo_ia")[["burnout_risk_score", "work_life_balance_score"]].mean().round(2)

,burnout_risk_score,work_life_balance_score
tramo_ia,,
Bajo,8.34,4.75
Medio-bajo,8.30,4.79
Medio-alto,8.34,4.84
Alto,8.45,4.51


Como la media de burnout está aplastada por el tope de 10, se mira también el porcentaje de cada nivel de burnout dentro de cada tramo. Con `normalize="index"` cada fila suma 100.

In [21]:
pd.crosstab(df_empleados_2["tramo_ia"], df_empleados_2["burnout_risk_level"], normalize="index") * 100

burnout_risk_level,High,Low,Medium
tramo_ia,,,
Bajo,73.840770,3.412073,22.747157
Medio-bajo,71.801802,3.153153,25.045045
Medio-alto,72.552448,2.622378,24.825175
Alto,75.430644,0.543971,24.025385


##### Paso 3: ¿Qué pesa más? Comparación con otras variables

Se calcula con qué variables se relaciona más el burnout.

In [22]:
df_empleados_2.corr(numeric_only=True)["burnout_risk_score"].sort_values()

work_life_balance_score        -0.846410
productivity_score             -0.392000
focus_hours_per_day            -0.160131
learning_time_hours_per_week   -0.025533
experience_years               -0.025427
task_complexity_score           0.005337
tasks_automated_percent         0.010333
error_rate_percent              0.011018
ai_tool_usage_hours_per_week    0.037898
collaboration_hours_per_week    0.192204
manual_work_hours_per_week      0.227832
meeting_hours_per_week          0.229863
burnout_risk_score              1.000000
Name: burnout_risk_score, dtype: float64

Se compara cuánto cambia el burnout entre tramos de IA y cuánto entre niveles de presión de plazos. La tercera columna muestra si la gente con más presión usa más IA.

In [23]:
df_empleados_2.groupby("deadline_pressure_level")[["burnout_risk_score", "work_life_balance_score", "ai_tool_usage_hours_per_week"]].mean().round(2)

,burnout_risk_score,work_life_balance_score,ai_tool_usage_hours_per_week
deadline_pressure_level,,,
High,9.97,3.29,10.34
Low,6.27,6.17,10.32
Medium,8.87,4.69,10.39


##### Conclusión empleados

- Entre el tramo Bajo y el Alto el burnout medio apenas cambia: pasa de 8,34 a 8,45 y su correlación con las horas de IA es de 0,038, prácticamente nula.
- El porcentaje de empleados con burnout alto tampoco sigue una tendencia clara (73,8 % en Bajo, 71,8 % en Medio-bajo, 72,6 % en Medio-alto y 75,4 % en Alto).
- El equilibrio vida-trabajo baja algo en el tramo Alto (de 4,84 en Medio-alto a 4,51), pero la diferencia es pequeña y su correlación con las horas de IA es de -0,063.
- La presión de plazos sí marca diferencias:
    - El burnout medio es de 6,27 con presión baja, 8,87 con media y 9,97 con alta.
    - El equilibrio vida-trabajo baja de 6,17 a 4,69 y a 3,29.
    - Las horas de IA son casi iguales en los tres niveles (unas 10,3 h de media), así que el mayor burnout con presión alta no coincide con un mayor uso de IA.
- La variable que más se relaciona con el burnout es el equilibrio vida-trabajo (-0,85), muy por delante de las horas de IA (0,04). Esas dos medidas son prácticamente la misma información.

## Dataset Estudiantes

##### Paso 1: Crear los tramos

Se crean los tramos de horas de IA igual que en empleados, con `qcut()` y se miran sus límites y su tamaño.

In [24]:
df_estudiantes["tramo_ia"] = pd.qcut(df_estudiantes["Weekly_GenAI_Hours"], q=4, labels=["Bajo", "Medio-bajo", "Medio-alto", "Alto"])

df_estudiantes.groupby("tramo_ia")["Weekly_GenAI_Hours"].agg(["min", "max", "count"])

,min,max,count
tramo_ia,,,
Bajo,0.00,2.39,12510
Medio-bajo,2.40,5.80,12516
Medio-alto,5.81,11.72,12479
Alto,11.73,40.00,12495


Los cuatro tramos tienen un tamaño casi idéntico (entre 12.479 y 12.516 estudiantes):

- **Bajo:** de 0 a 2,39 h a la semana
- **Medio-bajo:** de 2,40 a 5,80 h
- **Medio-alto:** de 5,81 a 11,72 h
- **Alto:** de 11,73 a 40 h

##### Paso 2: Burnout y ansiedad por tramo

El burnout es una categoría (Low, Medium, High), así que se miran porcentajes por tramo. La ansiedad va de 1 a 10.

In [25]:
pd.crosstab(df_estudiantes["tramo_ia"], df_estudiantes["Burnout_Risk_Level"], normalize="index") * 100

Burnout_Risk_Level,High,Low,Medium
tramo_ia,,,
Bajo,9.064748,49.664269,41.270983
Medio-bajo,12.927453,41.914350,45.158198
Medio-alto,22.157224,29.866175,47.976601
Alto,55.782313,9.467787,34.749900


In [26]:
df_estudiantes.groupby("tramo_ia")["Anxiety_Level_During_Exams"].mean().round(2)

tramo_ia
Bajo          3.81
Medio-bajo    3.90
Medio-alto    4.21
Alto          5.16
Name: Anxiety_Level_During_Exams, dtype: float64

##### Conclusión estudiantes

- El porcentaje de estudiantes con burnout alto crece en cada tramo: 9,06 %, 12,93 %, 22,16 % y 55,78 % en el Alto.
- El de burnout bajo cae del 49,66 % al 9,47 %. El salto grande se produce en el tramo Alto (más de 11,73 h a la semana).
- La ansiedad en los exámenes también sube: de 3,81 en el tramo Bajo a 5,16 en el Alto (3,90 y 4,21 en los intermedios).

#### Conclusión de la hipótesis 2

**La hipótesis se cumple solo en estudiantes** porque en ellos el burnout alto pasa del 9,06 % al 55,78 % y la ansiedad de 3,81 a 5,16 con más horas de IA, mientras que en empleados el burnout apenas cambia (de 8,34 a 8,45) y depende sobre todo de la presión de plazos.

- **Estudiantes:**
    - El burnout alto pasa del 9,06 % al 55,78 % y la ansiedad de 3,81 a 5,16 entre el tramo Bajo y el Alto.
    - El uso excesivo (más de 11,73 h a la semana) se asocia con peor bienestar.
- **Empleados:**
    - El burnout casi no cambia con las horas de IA (8,34 frente a 8,45; correlación de 0,038).
    - Lo que se relaciona con el burnout es la presión de plazos y el equilibrio vida-trabajo, no la IA.
- Con estos datos no se puede saber por qué la tendencia es distinta en cada grupo.

**Limitaciones:**

- En empleados el burnout está cortado en 10 (37,4 % de los casos), lo que deja poca variación para ver diferencias.
- Las escalas no son comparables (puntuación en empleados, categorías en estudiantes): solo se compara la tendencia.
- Correlación no es causalidad: no se sabe si usar más IA provoca burnout o si los estudiantes con más estrés usan más IA.
- Los datos son sintéticos.

**Observación sobre los datos:** La ficha de los estudiantes dice que variables como el burnout se generaron de forma uniforme, pero aquí cambia claramente con las horas de IA. La descripción no coincide del todo con los datos.

# Hipótesis 3: Los estudiantes que usan más IA sacan mejores notas, pero retienen menos

##### Paso 1: Crear la mejora

La nota final por sí sola no dice si el alumno mejoró, porque cada uno parte de un nivel. Se crea la variable `mejora_nota` (nota final menos nota inicial).

In [27]:
df_estudiantes["mejora_nota"] = df_estudiantes["Post_Semester_GPA"] - df_estudiantes["Pre_Semester_GPA"]
df_estudiantes["mejora_nota"].describe()

count    50000.000000
mean         0.203197
std          0.187192
min         -0.924000
25%          0.087000
50%          0.204000
75%          0.325000
max          1.008000
Name: mejora_nota, dtype: float64

- La nota sube de media 0,20 puntos tras el semestre (mediana 0,20).
- La mitad de los estudiantes mejora entre 0,09 y 0,33 puntos.
- Como el percentil 25 es positivo, al menos tres de cada cuatro mejoran.
- Hay quien empeora (mínimo de -0,92) y quien mejora más de un punto (máximo de 1,01), con una desviación de 0,19.

##### Paso 2: Correlación

Valores cerca de 0 significan relación débil. Pero solo mide líneas rectas, así que hacen falta los grupos.

In [28]:
df_estudiantes[["Weekly_GenAI_Hours", "mejora_nota", "Skill_Retention_Score"]].corr()

,Weekly_GenAI_Hours,mejora_nota,Skill_Retention_Score
Weekly_GenAI_Hours,1.000000,-0.046479,-0.118099
mejora_nota,-0.046479,1.000000,0.195833
Skill_Retention_Score,-0.118099,0.195833,1.000000


- La correlación de las horas de IA con la mejora de nota es de -0,05 y con la retención de habilidades, de -0,12. Las dos son muy débiles y de signo negativo.
- A primera vista, usar más IA no se asocia con mejorar más las notas y sí, ligeramente, con retener menos.
- Como la correlación solo detecta relaciones en línea recta, se comprueba con grupos en el paso siguiente.

##### Paso 3: Por tramo

Se calcula la mejora de nota y la retención medias de cada tramo y cuánto cambian de un tramo al siguiente.

In [29]:
medias_estudiantes = df_estudiantes.groupby("tramo_ia")[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)
medias_estudiantes

,mejora_nota,Skill_Retention_Score
tramo_ia,,
Bajo,0.188,75.653
Medio-bajo,0.206,76.429
Medio-alto,0.229,77.117
Alto,0.190,73.995


In [30]:
medias_estudiantes.diff()

,mejora_nota,Skill_Retention_Score
tramo_ia,,
Bajo,NaN,NaN
Medio-bajo,0.018,0.776
Medio-alto,0.023,0.688
Alto,-0.039,-3.122


- La mejora de nota sube del tramo Bajo al Medio-alto (0,188, 0,206 y 0,229) y baja en el Alto (0,190), casi igual que en el Bajo.
- La retención de habilidades sigue el mismo patrón: sube de 75,65 a 76,43 y a 77,12, y cae a 73,99 en el Alto, 3,12 puntos menos que en el tramo anterior y 1,66 menos que en el Bajo.
- Un uso moderado se asocia con algo más de mejora y de retención y el uso alto con una caída de la retención y sin ganancia de nota respecto a quien usa poca IA.
- Las diferencias son pequeñas: Entre Bajo y Medio-alto la mejora es de 0,04 puntos sobre 4.

##### Paso 4: Efecto techo

El 9,6 % de los alumnos acaba con un 4,0 (como se vio en la limpieza) y quien ya parte de una nota alta apenas puede mejorar. Para comprobar si eso distorsiona los resultados, se repite el paso 3 sin el 25 % con mejor nota inicial (el corte de 3,5 es aproximadamente el percentil 75 de la nota inicial).

In [31]:
sin_techo = df_estudiantes[df_estudiantes["Pre_Semester_GPA"] < 3.5]

# número de estudiantes que se quedan y porcentaje sobre el total
print("Estudiantes que se quedan:", len(sin_techo))
print("Porcentaje:", round(len(sin_techo) / len(df_estudiantes) * 100, 1))

Estudiantes que se quedan: 36654
Porcentaje: 73.3


In [32]:
sin_techo.groupby("tramo_ia")[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)

,mejora_nota,Skill_Retention_Score
tramo_ia,,
Bajo,0.201,75.181
Medio-bajo,0.218,75.814
Medio-alto,0.245,76.620
Alto,0.206,73.524


- Se repite el análisis sin los estudiantes con nota inicial de 3,5 o más, que tienen menos margen para mejorar. Quedan 36.654 estudiantes (73,3 %).
- El resultado es el mismo patrón: la mejora de nota pasa de 0,201 en Bajo a 0,218 en Medio-bajo, 0,245 en Medio-alto y 0,206 en Alto.
- La retención pasa de 75,18 a 75,81, 76,62 y 73,52.

Por tanto, el efecto techo no parece explicar lo observado en el paso 3.

##### Paso 5: ¿Importa más cómo se usa?

Se mira si el tipo de uso (`Primary_Use_Case`, para qué usa la IA cada estudiante) importa más que las horas.

In [33]:
medias_uso = df_estudiantes.groupby("Primary_Use_Case")[["mejora_nota", "Skill_Retention_Score"]].mean().round(3)
medias_uso

,mejora_nota,Skill_Retention_Score
Primary_Use_Case,,
Copywriting/Drafting,0.200,75.231
Debugging/Troubleshooting,0.249,78.069
Direct_Answer_Generation,0.133,73.719
Ideation,0.200,75.520
Summarizing_Reading,0.197,75.225


In [34]:
# distancia entre el mejor y el peor tipo de uso
medias_uso.max() - medias_uso.min()

mejora_nota              0.116
Skill_Retention_Score    4.350
dtype: float64

In [35]:
# distancia entre el mejor y el peor tramo de horas
medias_estudiantes.max() - medias_estudiantes.min()

mejora_nota              0.041
Skill_Retention_Score    3.122
dtype: float64

- Se compara el tipo de uso principal:
    - Los estudiantes que la usan para depurar código tienen la mayor mejora de nota (0,249) y la mayor retención (78,07).
    - Quienes piden respuestas directas tienen la menor mejora (0,133) y la menor retención (73,72).
    - Los otros tres usos (redactar, generar ideas y resumir lecturas) quedan en una zona intermedia, con una mejora de en torno a 0,20 y una retención de 75,2 a 75,5.

    
- La diferencia entre el mejor y el peor uso es de 0,116 puntos de mejora y 4,35 de retención. Entre el mejor y el peor tramo de horas, la diferencia es de 0,041 puntos de mejora y 3,12 de retención.
- Es decir, el tipo de uso se asocia con diferencias mayores que las horas de uso, sobre todo en la mejora de nota. Hay que tener en cuenta que se comparan 5 tipos de uso con 4 tramos de horas.

#### Conclusión de la hipótesis 3

**La hipótesis se cumple a medias**, porque el uso alto se asocia con menos retención (de 75,65 en Bajo a 73,99 en Alto), pero las notas no mejoran con más uso: la mejora es casi igual en Bajo y en Alto (0,188 y 0,190) y solo sube con un uso moderado (0,229 en Medio-alto).

- **Notas:** La mejora media pasa de 0,188 en el tramo Bajo a 0,190 en el Alto, prácticamente igual. Sube con un uso moderado (0,229 en Medio-alto, 0,04 puntos más que en Bajo) y cae con un uso alto. El uso intensivo no se asocia con mejores notas.
- **Retención:** Pasa de 75,65 en Bajo a 73,99 en Alto, 3,12 puntos menos que en Medio-alto. Retener menos solo aparece con un uso alto.
- **Las dos cosas a la vez no se dan:** Donde más mejora la nota (uso moderado) también es mayor la retención y donde baja la retención (uso alto) la nota tampoco mejora. No se observa "mejores notas pero menos aprendizaje".
- **El patrón se mantiene** al quitar a los estudiantes con nota inicial alta (paso 4), así que no depende del efecto techo.
- **Importa más cómo se usa que cuánto:** el uso para depurar código se asocia con la mayor mejora y retención y pedir respuestas directas con la menor (paso 5).

**Limitaciones:**

- La retención de habilidades es solo un indicador del aprendizaje.
- Una correlación no demuestra causalidad y el paso 5 no cruza el tipo de uso con las horas.
- Los datos son sintéticos y, según la ficha de Kaggle, la nota final se calculó a partir de la nota inicial, el uso de IA y las horas de estudio. Por eso es esperable que aparezcan relaciones entre estas variables y las diferencias son pequeñas (la correlación de las horas con la mejora es de -0,05).

# Resumen de resultados

| Hipótesis | Resultado | Evidencia principal |
|---|---|---|
| 1. Quien usa más IA es más productivo | **Se cumple** | La productividad media sube de 55,9 (tramo Bajo) a 74,6 (tramo Alto), con un coste: más errores (1,80 % a 2,47 %). |
| 2. El uso excesivo de la IA tiene efectos negativos en el bienestar | **Se cumple solo en estudiantes** | En estudiantes, el burnout alto pasa del 9,06 % al 55,78 %. En empleados, el burnout apenas cambia (8,34 a 8,45) y depende de la presión de plazos. |
| 3. Los estudiantes que usan más IA sacan mejores notas, pero retienen menos | **Se cumple a medias** | La retención cae solo con uso alto (75,65 a 73,99) y la mejora de nota es casi igual en Bajo y Alto (0,188 y 0,190). |